# Генерация признаков v4

## Философия: baseline + точечные улучшения

Baseline (1.6703) обошёл все последующие версии. Причины:
- **stride=14** (не 7 и не 30) — золотая середина: достаточно данных, умеренное перекрытие
- **single-fold training** (train N-1 → val N), а не cumulative
- **8 VALUE_COLS** с 3 агрегатами (sum, max, mean — нет std!)
- **Простые cross-features**: 3 тренда + 2 конверсии + 2 AOV

### Изменения v4 vs baseline:
1. Добавлены окна **270d и 365d** (используем всю историю — 407 дней)
2. Добавлены VALUE_COLS: **gmv_search, gmv_cat, search, cat** (+4 колонки)
3. Добавлены frequency: **freq_cart, freq_cat, active_days** (по окнам)
4. Добавлены recency: **recency_cart_days, recency_cat_days, user_age_days**
5. Добавлены cross-features: **ещё 15 тщательно отобранных**
6. **stride=14** сохранён (как в baseline)
7. Сохраняется в `features_v4/`

In [ ]:
import polars as pl
import numpy as np
from pathlib import Path
from datetime import date, timedelta
from typing import Optional

DATA_DIR = Path("../data/raw")
FEATURES_DIR = Path("../data/processed/features_v4")
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 4
STRIDE_DAYS = 14
BATCH_SIZE = 50_000
TARGET_COL = "gmv"

VALUE_COLS = [
    "gmv", "searches", "to_cart", "to_ord",
    "search_to_cart", "search_to_ord",
    "cat_to_cart", "cat_to_ord",
    "gmv_search", "gmv_cat",
    "search", "cat",
]

WINDOWS = [
    ("3d",   2,   0),
    ("7d",   6,   0),
    ("14d",  13,  0),
    ("30d",  29,  0),
    ("60d",  59,  0),
    ("90d",  89,  0),
    ("180d", 179, 0),
    ("270d", 269, 0),
    ("365d", 364, 0),
]

print(f"VALUE_COLS: {len(VALUE_COLS)}")
print(f"WINDOWS: {len(WINDOWS)}")
print(f"Stride: {STRIDE_DAYS} дней")

VALUE_COLS: 12
WINDOWS: 9
Stride: 14 дней


In [ ]:
def generate_cv_anchor_dates(
    data: pl.DataFrame,
    prediction_horizon_days: int = 30,
    stride_days: int = STRIDE_DAYS,
    min_history_days: int = 365,
    n_folds: Optional[int] = None,
) -> list[date]:
    min_date = data["event_date"].min()
    max_date = data["event_date"].max()

    latest_anchor = max_date - timedelta(days=prediction_horizon_days)
    earliest_anchor = min_date + timedelta(days=min_history_days - 1)

    n_steps = (latest_anchor - earliest_anchor).days // stride_days
    all_anchors = [latest_anchor - timedelta(days=i * stride_days) for i in range(n_steps + 1)]
    all_anchors = sorted(all_anchors)

    if n_folds:
        return all_anchors[-n_folds:]
    return all_anchors

In [ ]:
def _build_feature_exprs(anchor_val: date, windows: list, value_cols: list) -> list[pl.Expr]:
    exprs = []
    
    for w_name, start_off, end_off in windows:
        w_start = anchor_val - timedelta(days=start_off)
        w_end = anchor_val - timedelta(days=end_off)
        mask = pl.col("event_date").is_between(w_start, w_end)

        for col in value_cols:
            exprs.append(pl.when(mask).then(pl.col(col)).otherwise(0.0).sum().alias(f"{col}_sum_{w_name}"))
            exprs.append(pl.when(mask).then(pl.col(col)).otherwise(None).max().alias(f"{col}_max_{w_name}"))
            exprs.append(pl.when(mask).then(pl.col(col)).otherwise(None).mean().alias(f"{col}_mean_{w_name}"))
            
        exprs.append(pl.when(mask & (pl.col("searches") > 0)).then(1).otherwise(0).sum().alias(f"freq_search_{w_name}"))
        exprs.append(pl.when(mask & (pl.col("to_ord") > 0)).then(1).otherwise(0).sum().alias(f"freq_ord_{w_name}"))
        exprs.append(pl.when(mask & (pl.col("to_cart") > 0)).then(1).otherwise(0).sum().alias(f"freq_cart_{w_name}"))
        exprs.append(pl.when(mask & (pl.col("cat") > 0)).then(1).otherwise(0).sum().alias(f"freq_cat_{w_name}"))
        exprs.append(pl.when(mask & ((pl.col("searches") > 0) | (pl.col("cat") > 0))).then(1).otherwise(0).sum().alias(f"active_days_{w_name}"))

    last_ord_dt    = pl.col("event_date").filter(pl.col("to_ord") > 0).max()
    last_search_dt = pl.col("event_date").filter(pl.col("searches") > 0).max()
    last_cart_dt   = pl.col("event_date").filter(pl.col("to_cart") > 0).max()
    last_cat_dt    = pl.col("event_date").filter(pl.col("cat") > 0).max()
    first_event_dt = pl.col("event_date").min()
    
    exprs.append((pl.lit(anchor_val) - last_ord_dt).dt.total_days().fill_null(-1).alias("recency_ord_days"))
    exprs.append((pl.lit(anchor_val) - last_search_dt).dt.total_days().fill_null(-1).alias("recency_search_days"))
    exprs.append((pl.lit(anchor_val) - last_cart_dt).dt.total_days().fill_null(-1).alias("recency_cart_days"))
    exprs.append((pl.lit(anchor_val) - last_cat_dt).dt.total_days().fill_null(-1).alias("recency_cat_days"))
    exprs.append((pl.lit(anchor_val) - first_event_dt).dt.total_days().fill_null(-1).alias("user_age_days"))
    
    return exprs

In [ ]:
def _build_secondary_exprs() -> list[pl.Expr]:
    eps = 1.0
    exprs = []
    
    exprs.extend([
        (pl.col("gmv_sum_7d") / (pl.col("gmv_sum_30d") + eps)).alias("trend_gmv_7d_30d"),
        (pl.col("searches_sum_7d") / (pl.col("searches_sum_30d") + eps)).alias("trend_search_7d_30d"),
        (pl.col("gmv_sum_30d") / (pl.col("gmv_sum_90d") + eps)).alias("trend_gmv_30d_90d"),
        (pl.col("gmv_sum_14d") / (pl.col("gmv_sum_30d") + eps)).alias("trend_gmv_14d_30d"),
        (pl.col("gmv_sum_30d") / (pl.col("gmv_sum_180d") + eps)).alias("trend_gmv_30d_180d"),
        (pl.col("gmv_sum_90d") / (pl.col("gmv_sum_365d") + eps)).alias("trend_gmv_90d_365d"),
        (pl.col("to_ord_sum_7d") / (pl.col("to_ord_sum_30d") + eps)).alias("trend_ord_7d_30d"),
    ])
    
    exprs.extend([
        (pl.col("search_to_cart_sum_30d") / (pl.col("searches_sum_30d") + eps)).alias("cr_search_to_cart_30d"),
        (pl.col("to_ord_sum_30d") / (pl.col("to_cart_sum_30d") + eps)).alias("cr_cart_to_ord_30d"),
        (pl.col("search_to_ord_sum_30d") / (pl.col("searches_sum_30d") + eps)).alias("cr_search_to_ord_30d"),
        (pl.col("cat_to_cart_sum_30d") / (pl.col("cat_sum_30d") + eps)).alias("cr_cat_to_cart_30d"),
        (pl.col("cat_to_ord_sum_30d") / (pl.col("cat_sum_30d") + eps)).alias("cr_cat_to_ord_30d"),
    ])
    
    exprs.extend([
        (pl.col("gmv_sum_30d") / (pl.col("to_ord_sum_30d") + eps)).alias("aov_30d"),
        (pl.col("gmv_sum_90d") / (pl.col("to_ord_sum_90d") + eps)).alias("aov_90d"),
        (pl.col("gmv_sum_180d") / (pl.col("to_ord_sum_180d") + eps)).alias("aov_180d"),
    ])
    
    exprs.extend([
        (pl.col("gmv_sum_30d") / (pl.col("active_days_30d") + eps)).alias("gmv_per_active_day_30d"),
        (pl.col("gmv_sum_90d") / (pl.col("active_days_90d") + eps)).alias("gmv_per_active_day_90d"),
        (pl.col("gmv_cat_sum_30d") / (pl.col("gmv_search_sum_30d") + pl.col("gmv_cat_sum_30d") + eps)).alias("ratio_gmv_cat_30d"),
    ])
    
    return exprs

In [ ]:
def generate_features_and_targets(
    data: pl.DataFrame,
    anchor: date,
    user_batch: list[int],
    is_train: bool = True
) -> pl.DataFrame:
    
    max_back = max(w[1] for w in WINDOWS)
    
    data_f = data.filter(
        pl.col("user_id").is_in(user_batch) & 
        (pl.col("event_date") <= anchor) & 
        (pl.col("event_date") >= anchor - timedelta(days=max_back))
    )

    if len(data_f) > 0:
        features = (
            data_f.group_by("user_id")
            .agg(_build_feature_exprs(anchor, WINDOWS, VALUE_COLS))
            .with_columns(_build_secondary_exprs())
            .with_columns(anchor_date=pl.lit(anchor))
        )
    else:
        features = pl.DataFrame({"user_id": user_batch, "anchor_date": anchor})
    
    index_df = pl.DataFrame({"user_id": user_batch}).with_columns(anchor_date=pl.lit(anchor))
    result = index_df.join(features, on=["anchor_date", "user_id"], how="left")
    
    feat_cols = [c for c in result.columns if c not in ["anchor_date", "user_id", "recency_ord_days", "recency_search_days", "recency_cart_days", "recency_cat_days"]]
    result = result.with_columns([pl.col(c).fill_null(0.0) for c in feat_cols])

    if is_train:
        t_start = anchor + timedelta(days=1)
        t_end   = anchor + timedelta(days=30)
        targets = (
            data.filter(
                pl.col("user_id").is_in(user_batch) & 
                pl.col("event_date").is_between(t_start, t_end)
            )
            .group_by("user_id")
            .agg(pl.col(TARGET_COL).sum().alias("target"))
        )
        result = result.join(targets, on="user_id", how="left").with_columns(pl.col("target").fill_null(0.0))
    else:
        result = result.with_columns(pl.lit(None).cast(pl.Float64).alias("target"))
        
    return result

In [ ]:
import time

print("Читаем сырые данные...")
data = pl.read_parquet(DATA_DIR / 'train.parquet')
user_ids = data["user_id"].unique().sort().to_list()
n_batches = (len(user_ids) + BATCH_SIZE - 1) // BATCH_SIZE

anchors_time_folds = generate_cv_anchor_dates(data, n_folds=N_FOLDS)
anchor_end_of_time = data["event_date"].max()

print(f"Пользователей: {len(user_ids):,}")
print(f"Период данных: {data['event_date'].min()} — {data['event_date'].max()}")
print(f"Фолды: {len(anchors_time_folds)} ({anchors_time_folds[0]} → {anchors_time_folds[-1]})")
print(f"Тест: {anchor_end_of_time}")
print(f"Батчей: {n_batches}")

t0 = time.time()

for fold_idx, anchor in enumerate(anchors_time_folds):
    fold_dir = FEATURES_DIR / f"fold_{fold_idx:02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    
    for batch in range(n_batches):
        current_batch = user_ids[batch * BATCH_SIZE : (batch + 1) * BATCH_SIZE]
        out_df = generate_features_and_targets(data, anchor, current_batch, is_train=True)
        out_df.write_parquet(fold_dir / f"batch_{batch:04d}.parquet")
        
    elapsed = time.time() - t0
    print(f"  fold_{fold_idx:02d} ({anchor}) — {elapsed:.0f}s")

# Тест
fold_dir = FEATURES_DIR / "fold_test"
fold_dir.mkdir(parents=True, exist_ok=True)

for batch in range(n_batches):
    current_batch = user_ids[batch * BATCH_SIZE : (batch + 1) * BATCH_SIZE]
    out_df = generate_features_and_targets(data, anchor_end_of_time, current_batch, is_train=False)
    out_df.write_parquet(fold_dir / f"batch_{batch:04d}.parquet")

elapsed = time.time() - t0
print(f"  fold_test ({anchor_end_of_time}) — {elapsed:.0f}s")

test_check = pl.read_parquet(FEATURES_DIR / "fold_test" / "batch_*.parquet")
drop_cols = ["user_id", "anchor_date", "target"]
features_list = [c for c in test_check.columns if c not in drop_cols]
print(f"\nИтого фичей: {len(features_list)}")
print(f"Размер fold_test: {test_check.shape}")
print(f"Общее время: {elapsed:.0f}s")

Читаем сырые данные...
Пользователей: 250,000
Период данных: 2025-01-01 — 2026-02-13
Фолды: 2 (2025-12-31 → 2026-01-14)
Тест: 2026-02-13
Батчей: 5
  fold_00 (2025-12-31) — 33s
  fold_01 (2026-01-14) — 67s
  fold_test (2026-02-13) — 98s

Итого фичей: 392
Размер fold_test: (250000, 395)
Общее время: 98s
